<a href="https://colab.research.google.com/github/nicoleiliuk/DataScience_Nicole/blob/main/DS_Prediction_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bat / Non-bat Prediction Tool

This notebook applies the trained Random Forest model to new acoustic recordings.

Researchers only need to:

1. Run the notebook cells in order.
2. Upload one or more `.wav` recordings.
3. Review the resulting classification table.
4. Download the final CSV.

Each recording is assigned to one of three screening categories:

- **BAT** — high-confidence bat prediction.
- **NON-BAT** — high-confidence non-bat prediction.
- **EXPERT REVIEW** — intermediate-confidence recording that should be inspected manually.

This tool is intended for acoustic screening and does not replace expert identification.


## 1. Install required libraries

In [ ]:
!pip -q install librosa soundfile scikit-learn joblib


## 2. Connect to Google Drive

The trained model bundle is expected at:

```text
MyDrive/bat_project/outputs/bat_classifier_model.joblib
```


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Import libraries and define paths

In [ ]:
from pathlib import Path

import joblib
import librosa
import numpy as np
import pandas as pd

from google.colab import files

PROJECT_DIR = Path("/content/drive/MyDrive/bat_project")
MODEL_DIR = PROJECT_DIR / "outputs"
MODEL_PATH = MODEL_DIR / "bat_classifier_model.joblib"

UPLOAD_DIR = Path("/content/new_recordings")
OUTPUT_DIR = Path("/content/prediction_outputs")

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model file exists:", MODEL_PATH.exists())
print("Upload directory:", UPLOAD_DIR)
print("Output directory:", OUTPUT_DIR)


## 4. Load the trained model bundle

The bundle contains:

- the trained Random Forest model;
- the exact feature-column order used during training;
- the target sample rate;
- the final confidence thresholds used for screening.

The current training pipeline exports thresholds of **0.30 for NON-BAT** and **0.70 for BAT**.


In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model bundle not found at: {MODEL_PATH}\n"
        "Run the model-saving cell in the training notebook first."
    )

model_bundle = joblib.load(MODEL_PATH)

rf_model = model_bundle["model"]

#Feature columns
if "feature_columns" in model_bundle:
    feature_columns = model_bundle["feature_columns"]

elif hasattr(rf_model, "feature_names_in_"):
    feature_columns = list(rf_model.feature_names_in_)
    print(
        "Warning: feature_columns were not stored in the bundle. "
        "Recovered them from the trained model."
    )

else:
    raise KeyError(
        "The model bundle does not contain 'feature_columns', "
        "and they could not be recovered from the trained model. "
        "Re-save the model bundle from the training notebook."
    )

#All recordings are resampled to 256 kHz before feature extraction.
TARGET_SAMPLE_RATE = model_bundle.get(
    "target_sample_rate",
    256000
)

NON_BAT_THRESHOLD = model_bundle.get(
    "non_bat_threshold",
    0.30
)

BAT_THRESHOLD = model_bundle.get(
    "bat_threshold",
    0.70
)

print("Model loaded successfully.")
print(f"Target sample rate: {TARGET_SAMPLE_RATE} Hz")
print(f"Number of expected features: {len(feature_columns)}")
print(f"NON-BAT threshold: P(bat) <= {NON_BAT_THRESHOLD:.2f}")
print(f"BAT threshold: P(bat) >= {BAT_THRESHOLD:.2f}")
print(
    f"EXPERT REVIEW: "
    f"{NON_BAT_THRESHOLD:.2f} < P(bat) < {BAT_THRESHOLD:.2f}"
)

## 5. Define audio preprocessing

This function must remain identical to the preprocessing used during training.


In [ ]:
#Load and minimally preprocess one acoustic recording
def preprocess_audio(audio_path, normalize=True):

#Load the original recording without automatic resampling.
    signal, original_sample_rate = librosa.load(
        audio_path,
        sr=None,
        mono=True
    )
#Standardize recordings to the sample rate used during model training.
    if original_sample_rate != TARGET_SAMPLE_RATE:
        signal = librosa.resample(
            signal,
            orig_sr=original_sample_rate,
            target_sr=TARGET_SAMPLE_RATE
        )

    signal = np.nan_to_num(
        signal,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )
#Remove DC offset before feature extraction.
    if len(signal) > 0:
        signal = signal - np.mean(signal)

    peak_amplitude = float(np.max(np.abs(signal))) if len(signal) > 0 else 0.0

#Peak normalization reduces amplitude differences among recordings.
    if normalize and peak_amplitude > 0:
        signal = signal / peak_amplitude

    processing_info = {
        "original_sample_rate": int(original_sample_rate),
        "processed_sample_rate": int(TARGET_SAMPLE_RATE),
        "duration_seconds": float(len(signal) / TARGET_SAMPLE_RATE),
        "original_peak_amplitude": peak_amplitude
    }

    return signal, TARGET_SAMPLE_RATE, processing_info


## 6. Define acoustic feature extraction

This function must also remain identical to the function used during model training.


In [ ]:
#Extract the same acoustic features used during model training.
#Do not change feature definitions or parameters without retraining the model.
def extract_features(signal, sample_rate):

    if len(signal) == 0:
        raise ValueError("The audio signal is empty.")

    features = {}
    n_fft = 2048
    hop_length = 512

    mfcc = librosa.feature.mfcc(
        y=signal, sr=sample_rate, n_mfcc=20, n_mels=40,
        n_fft=n_fft, hop_length=hop_length
    )

    for i in range(20):
        features[f"mfcc_{i + 1}_mean"] = float(np.mean(mfcc[i]))
        features[f"mfcc_{i + 1}_std"] = float(np.std(mfcc[i]))

    centroid = librosa.feature.spectral_centroid(
        y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length
    )
    features["spectral_centroid_mean"] = float(np.mean(centroid))
    features["spectral_centroid_std"] = float(np.std(centroid))

    bandwidth = librosa.feature.spectral_bandwidth(
        y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length
    )
    features["spectral_bandwidth_mean"] = float(np.mean(bandwidth))
    features["spectral_bandwidth_std"] = float(np.std(bandwidth))

    rolloff = librosa.feature.spectral_rolloff(
        y=signal, sr=sample_rate, n_fft=n_fft, hop_length=hop_length,
        roll_percent=0.85
    )
    features["spectral_rolloff_mean"] = float(np.mean(rolloff))
    features["spectral_rolloff_std"] = float(np.std(rolloff))

    flatness = librosa.feature.spectral_flatness(
        y=signal, n_fft=n_fft, hop_length=hop_length
    )
    features["spectral_flatness_mean"] = float(np.mean(flatness))
    features["spectral_flatness_std"] = float(np.std(flatness))

    rms = librosa.feature.rms(
        y=signal, frame_length=n_fft, hop_length=hop_length
    )
    features["rms_mean"] = float(np.mean(rms))
    features["rms_std"] = float(np.std(rms))

    zcr = librosa.feature.zero_crossing_rate(
        y=signal, frame_length=n_fft, hop_length=hop_length
    )
    features["zcr_mean"] = float(np.mean(zcr))
    features["zcr_std"] = float(np.std(zcr))

    return features


## 7. Upload new recordings

Select one or more `.wav` files. Previously uploaded files in the temporary Colab folder are removed before each new upload.


In [ ]:
#Run the cell first to be able to upload files
for existing_file in UPLOAD_DIR.glob("*"):
    if existing_file.is_file():
        existing_file.unlink()

uploaded_files = files.upload()

for filename, file_content in uploaded_files.items():
    destination = UPLOAD_DIR / Path(filename).name
    destination.write_bytes(file_content)

wav_files = sorted(UPLOAD_DIR.glob("*.wav"))

print(f"Uploaded WAV files: {len(wav_files)}")

if not wav_files:
    raise ValueError("No .wav files were uploaded.")

for audio_path in wav_files[:10]:
    print("-", audio_path.name)


## 8. Classify the uploaded recordings

For every uploaded recording, the notebook:
- applies the same preprocessing used during model training;
- extracts the same 52 acoustic features;
- arranges the features in the exact order expected by the trained model;
- calculates the Random Forest probability assigned to the `bat` class;
- applies the confidence-based screening policy.

The screening thresholds are read directly from the saved model bundle:

- `P(bat) <= 0.30` → **NON-BAT**
- `0.30 < P(bat) < 0.70` → **EXPERT REVIEW**
- `P(bat) >= 0.70` → **BAT**


In [ ]:
def assign_review_category(probability_bat):
    if probability_bat >= BAT_THRESHOLD:
        return "BAT"
    elif probability_bat <= NON_BAT_THRESHOLD:
        return "NON-BAT"
    else:
        return "EXPERT REVIEW"


if "bat" not in rf_model.classes_:
    raise ValueError(
        f"The loaded model does not contain a 'bat' class. "
        f"Available classes: {list(rf_model.classes_)}"
    )

bat_index = list(rf_model.classes_).index("bat")

prediction_rows = []
error_rows = []

audio_files = sorted(UPLOAD_DIR.glob("*.wav"))

if len(audio_files) == 0:
    raise FileNotFoundError(
        "No .wav files were found in the upload folder. "
        "Run the upload cell first."
    )

for audio_path in audio_files:

    try:
        signal, sample_rate, processing_info = preprocess_audio(audio_path)

        features = extract_features(
            signal,
            sample_rate
        )

        feature_row = pd.DataFrame([features])

        # Guarantee the same feature order used during training.
        missing_features = [
            column for column in feature_columns
            if column not in feature_row.columns
        ]

        if missing_features:
            raise ValueError(
                f"Missing expected features: {missing_features}"
            )

        feature_row = feature_row[feature_columns]

        predicted_class = rf_model.predict(feature_row)[0]

        probability_bat = rf_model.predict_proba(
            feature_row
        )[0, bat_index]

        review_status = assign_review_category(
            probability_bat
        )

        prediction_rows.append({
            "filename": audio_path.name,
            "model_prediction": predicted_class,
            "probability_bat": probability_bat,
            "review_status": review_status
        })

    except Exception as error:

        error_rows.append({
            "filename": audio_path.name,
            "error": str(error)
        })


predictions = pd.DataFrame(prediction_rows)

prediction_errors = pd.DataFrame(
    error_rows,
    columns=["filename", "error"]
)

if predictions.empty:
    print("No recordings were successfully classified.")
else:
    predictions = predictions.sort_values(
        "probability_bat",
        ascending=False
    ).reset_index(drop=True)

    display(predictions)

    print("\nScreening summary:")

    review_summary = (
        predictions["review_status"]
        .value_counts()
        .rename_axis("review_status")
        .reset_index(name="n")
    )

    review_summary["percentage"] = (
        review_summary["n"]
        / len(predictions)
        * 100
    )

    display(review_summary)

if not prediction_errors.empty:
    print("\nFiles that could not be processed:")
    display(prediction_errors)


## 9. Save and download the results

The final CSV contains:

- `filename`
- `model_prediction`
- `probability_bat`
- `review_status`

Files that could not be processed are saved separately when processing errors occur.


In [ ]:
PREDICTIONS_PATH = OUTPUT_DIR / "predictions.csv"

predictions.to_csv(PREDICTIONS_PATH, index=False)
print("Predictions saved to:", PREDICTIONS_PATH)

if not prediction_errors.empty:
    ERRORS_PATH = OUTPUT_DIR / "prediction_errors.csv"
    prediction_errors.to_csv(ERRORS_PATH, index=False)
    print("Errors saved to:", ERRORS_PATH)

files.download(str(PREDICTIONS_PATH))


## Interpretation

The `review_status` column should be used as the main screening output:

- **BAT:** `P(bat) >= 0.70`. The recording is retained as a high-confidence bat candidate.
- **NON-BAT:** `P(bat) <= 0.30`. The recording is considered a high-confidence non-bat candidate.
- **EXPERT REVIEW:** `0.30 < P(bat) < 0.70`. The model is not sufficiently confident for automatic screening, so the recording should be inspected manually.

The `model_prediction` column reports the Random Forest's standard binary decision and is included for reference. The confidence-based `review_status` is the recommended output for the screening workflow.

### Important limitations

This classifier was trained on the acoustic conditions represented in the training dataset. Validation showed that performance can decrease for recorder types or recording conditions that are poorly represented or absent from training.

Therefore:

- recordings from new recorder types or substantially different acoustic environments should be interpreted cautiously;
- **EXPERT REVIEW** recordings should always be inspected manually;
- the classifier should be treated as a screening tool rather than a replacement for expert acoustic identification;
- if the model is retrained using substantially different data, its confidence thresholds should be re-evaluated.
